# 15 · Query an authenticated HTTP gateway

Run the real local API lifecycle, then optionally connect the same notebook to your own HTTPS gateway.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/),
[CC0 license](https://github.com/allisonhorst/palmerpenguins/blob/8957207b78d6ccd1b4654a9dd9c9041b657478ab/DESCRIPTION).
Citation: Horst, Hill and Gorman (2020), *palmerpenguins: Palmer Archipelago
(Antarctica) penguin data*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
The helper downloads the pinned original CSV and verifies its SHA-256.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
penguins = lab.penguins()
config = lab.catalog(sources=[{"id": "penguins", "type": "csv", "path": str(penguins)}])
request = {"mode": "federated", "sources": ["penguins"], "sql": """
    SELECT species, COUNT(*)::BIGINT AS observations
    FROM penguins GROUP BY species ORDER BY species
"""}
with lab.gateway(config) as gateway:
    # The helper starts `kelvo serve` on loopback with a fresh secret token.
    # Do not print gateway: it contains that token.
    table = remote_query(gateway["url"], gateway["token"], request,
                         timeout=120, max_bytes=16 * 1024 * 1024)
expected = {"Adelie": 152, "Chinstrap": 68, "Gentoo": 124}
actual = dict(zip(table["species"].to_pylist(), table["observations"].to_pylist()))
assert actual == expected
display(table.to_pandas())
print("Authenticated submit, Arrow retrieval and successful terminal status verified.")


## Understand completion

Submission creates a handle. Fetching its results starts execution, and result
retrieval has one consumer. An HTTP 200 or partial Arrow stream alone is not
success. The helper checks the terminal query state after consuming the result.

The local listener is a single trust domain. Its token authorizes its configured
catalog; this example does not establish per-user row-level access or a shared
tenant security boundary.


## Optional: use your own gateway

Leave the variables unset to keep the notebook self-contained. To opt in, set
`KELVO_REMOTE_URL` to your HTTPS endpoint and `KELVO_REMOTE_SOURCE` to an alias
your operator permits. The token comes from a hidden prompt and is never saved
in the notebook. Use a dedicated read-only test catalog.

The client validates TLS and refuses credential-bearing redirects. Do not put
tokens in URLs, notebook cells, output, git history or shared runtime metadata.


In [ ]:
import getpass
import re
remote_url = os.environ.get("KELVO_REMOTE_URL", "").strip()
if remote_url:
    assert remote_url.startswith("https://"), "Remote examples require HTTPS"
    remote_source = os.environ.get("KELVO_REMOTE_SOURCE", "").strip()
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", remote_source), "Set a plain registered source alias"
    remote_token = getpass.getpass("Kelvo bearer token: ")
    try:
        remote_table = remote_query(remote_url, remote_token, {
            "mode": "federated", "sources": [remote_source],
            "sql": f'SELECT * FROM "{remote_source}" LIMIT 10',
        }, timeout=120, max_bytes=16 * 1024 * 1024)
        assert remote_table.num_rows <= 10
        display(remote_table.to_pandas())
    finally:
        del remote_token
else:
    print("Remote step skipped. The real authenticated local gateway test above completed.")


For an accelerated alias, use its configured name. Database attachments may
require a schema-qualified table rather than `SELECT * FROM alias`; adapt the
SQL to the trusted catalog. Native requests instead use `mode: native` and one
`connection_id`. See the [API guide](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/usage.md#http-api)
and [production deployment](https://github.com/SYNEHQ/kelvo-go/blob/cargo/deploy/README.md).


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
